# Pengujian Validitas dan Reliabilitas Instrumen

**Probabilitas dan Statistika** · M. Narum Bella (2455301225) · 3 TI F · Politeknik Caltex Riau

Data: respons Google Form **"Kuesioner Penggunaan AI dan Lifestyle Media Sosial Mahasiswa"** dari
**kelas 3 TI F** (`Dataset_Kuesioner_3TIF.xlsx`, jawaban masih berupa label seperti "Setuju (S)").

Langkah 1-8 mengikuti modul. Satu penyesuaian terhadap data: kolom "durasi rata-rata penggunaan gadget"
berada sesudah kolom demografi tetapi bukan butir Likert (isinya "4 – 7 jam", dst.), sehingga dikeluarkan
dari analisis.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NarumBella-TIG/UjiValiditas/blob/main/uji_validitas_reliabilitas.ipynb)

## Langkah 1: Persiapan lingkungan dan impor pustaka

In [1]:
import re
import pandas as pd
import numpy as np
from scipy import stats

## Langkah 2: Unggah berkas data
Di Google Colab akan muncul tombol **Choose Files**; pilih `Dataset_Kuesioner_3TIF.xlsx`.

In [2]:
try:
    from google.colab import files
    up = files.upload()
    nama_file = list(up.keys())[0]
except ImportError:                                   # jika dijalankan di luar Colab
    nama_file = "Dataset_Kuesioner_3TIF.xlsx"
print("File terbaca:", nama_file)

File terbaca: Dataset_Kuesioner_3TIF.xlsx


## Langkah 3: Membaca data dan memisahkan butir

In [3]:
JUMLAH_KOLOM_DEMOGRAFI = 5   # Timestamp, Kelas, Jenis Kelamin, 2 kolom jumlah platform AI

df = pd.read_excel(nama_file)
df = df[df["Kelas"].astype(str).str.strip() == "3 TI F"].reset_index(drop=True)   # analisis khusus 3 TI F
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

# Kolom durasi gadget bukan butir Likert (isinya "4 – 7 jam", dst.), jadi dikeluarkan
kolom_bukan_butir = [k for k in butir.columns if not re.match(r"^(.*?) \[", k)]
print("Kolom non-butir yang dikeluarkan:", kolom_bukan_butir)
butir = butir.drop(columns=kolom_bukan_butir)

print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))
print("Responden per kelas:", df["Kelas"].value_counts().to_dict())

Kolom non-butir yang dikeluarkan: ['Berapa durasi rata-rata kamu menggunakan gadget per hari, mulai dari kebutuhan harian sampai dengan hiburan?']
Jumlah responden : 30
Jumlah butir     : 29
Data kosong      : 0
Responden per kelas: {'3 TI F': 30}


## Langkah 4: Mengubah jawaban Likert menjadi angka
Label jawaban diubah menjadi angka: STS = 1, TS = 2, N = 3, S = 4, SS = 5.

In [4]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}

num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")

# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
print("Nilai yang muncul:", sorted(int(v) for v in pd.unique(num.values.ravel())))
# 5 responden pertama; judul kolom disingkat K1-K29 agar tabel muat di layar
num.head().set_axis([f"K{i}" for i in range(1, num.shape[1] + 1)], axis=1)

Sel yang gagal dikonversi: 0
Nilai yang muncul: [1, 2, 3, 4, 5]


,K1,K2,K3,K4,K5,K6,K7,K8,K9,K10,...,K20,K21,K22,K23,K24,K25,K26,K27,K28,K29
0,2,4,4,4,4,4,4,2,2,2,...,1,2,2,2,2,4,4,4,4,4
1,4,4,4,4,4,4,4,4,2,2,...,4,4,4,4,4,4,4,4,4,4
2,5,5,5,5,5,5,5,5,4,4,...,5,2,5,5,5,5,5,5,5,5
3,4,5,5,5,4,4,4,4,2,2,...,4,4,2,5,4,4,4,2,5,4
4,5,4,5,5,5,4,5,5,4,1,...,4,1,2,1,2,5,5,4,4,4


## Langkah 5: Mengelompokkan butir per dimensi

In [5]:
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?) \[", kolom).group(1).strip().rstrip(".")
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

# Kode butir tetap (B1, B2, ...) agar nomor tidak bergeser saat ada butir yang dibuang
KODE = {kolom: f"B{i}" for nama, daftar in dimensi.items() for i, kolom in enumerate(daftar, 1)}

# Daftar kode butir dan pernyataannya
kamus_butir = pd.DataFrame([
    {"Dimensi": nama, "Butir": f"B{i}", "Pernyataan": kolom[kolom.index("[") + 1:].rstrip("]")}
    for nama, daftar in dimensi.items() for i, kolom in enumerate(daftar, 1)
])
kamus_butir

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik: 4 butir
Respon emosional: 5 butir


,Dimensi,Butir,Pernyataan
0,Penggunaan AI pada tugas atau projek kuliah,B1,Saya menggunakan AI hampir setiap kali mendapa...
1,Penggunaan AI pada tugas atau projek kuliah,B2,Saya menggunakan AI untuk mencari penyelesaian...
2,Penggunaan AI pada tugas atau projek kuliah,B3,Saya menggunakan AI untuk mencari ide atau ins...
3,Penggunaan AI pada tugas atau projek kuliah,B4,Saya menggunakan AI untuk memahami materi kuli...
4,Penggunaan AI pada tugas atau projek kuliah,B5,Saya menggunakan AI untuk menyusun atau memper...
5,Penggunaan AI pada tugas atau projek kuliah,B6,Saya menggunakan AI untuk membuat atau memperb...
6,Penggunaan AI pada tugas atau projek kuliah,B7,Saya menggunakan lebih dari satu platform AI u...
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,Saya merasa perlu menggunakan AI setiap kali m...
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,Saya merasa kesulitan menyelesaikan tugas kuli...
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,Saya cenderung langsung mengikuti jawaban AI t...


## Langkah 6: Uji validitas (korelasi Pearson butir-total)

In [6]:
def r_tabel_untuk(n, alpha=0.05):
    t = stats.t.ppf(1 - alpha / 2, n - 2)
    return t / np.sqrt((n - 2) + t**2)


def uji_validitas(num, dimensi):
    n = len(num)
    r_tabel = r_tabel_untuk(n)
    hasil = []
    for nama, daftar in dimensi.items():
        x = num[daftar]
        total = x.sum(axis=1)
        for i, kolom in enumerate(daftar, 1):
            r, p = stats.pearsonr(x[kolom], total)
            r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
            hasil.append({
                "Dimensi": nama,
                "Butir": KODE[kolom],
                "r hitung": round(r, 3),
                "Corrected r": round(r_koreksi, 3),
                "p-value": round(p, 4),
                "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
            })
    return pd.DataFrame(hasil), r_tabel


tabel_validitas, r_tabel = uji_validitas(num, dimensi)
print(f"n = {len(num)} | r tabel (α=0,05; df={len(num) - 2}) = {r_tabel:.3f}")
print("Butir valid:", (tabel_validitas["Keterangan"] == "Valid").sum(), "dari", len(tabel_validitas))
tabel_validitas

n = 30 | r tabel (α=0,05; df=28) = 0.361
Butir valid: 29 dari 29


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.790,0.668,0.0000,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.691,0.585,0.0000,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.627,0.481,0.0002,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.554,0.424,0.0015,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.722,0.495,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.800,0.723,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.654,0.548,0.0001,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.710,0.605,0.0000,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.780,0.685,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.710,0.617,0.0000,Valid


## Langkah 7: Uji reliabilitas (Cronbach's alpha)

In [7]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))


def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")


def uji_reliabilitas(num, dimensi):
    hasil_reliabilitas, alpha_dihapus = [], []
    for nama, daftar in dimensi.items():
        x = num[daftar]
        a = cronbach_alpha(x)
        hasil_reliabilitas.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                                   "Cronbach's alpha": round(a, 3), "Kategori": kategori(a)})
        for i, kolom in enumerate(daftar, 1):
            if len(daftar) > 2:
                alpha_dihapus.append({"Dimensi": nama, "Butir": KODE[kolom],
                                      "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3)})
    return pd.DataFrame(hasil_reliabilitas), pd.DataFrame(alpha_dihapus)


tabel_reliabilitas, tabel_alpha_dihapus = uji_reliabilitas(num, dimensi)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.799,Reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.886,Baik
2,Kebiasaan penggunaan teknologi digital,6,0.890,Baik
3,Prokrastinasi akademik,4,0.860,Baik
4,Respon emosional,5,0.928,Sangat baik


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.744
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.768
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.782
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.791
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.808
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.746
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.775
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.879
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.869
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.877


## Langkah 8: Menggabungkan dan mengunduh hasil
Kolom tambahan "Keputusan" mengikuti tabel pedoman pengambilan keputusan pada modul (bagian 6).

In [8]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")
alpha_dimensi = tabel_reliabilitas.set_index("Dimensi")["Cronbach's alpha"]


def keputusan(baris):
    valid = baris["Keterangan"] == "Valid"
    naik = baris["Alpha jika dihapus"] > alpha_dimensi[baris["Dimensi"]]
    if valid and not naik:
        return "Pertahankan"
    if valid and naik:
        return "Tinjau isi butir"
    if not valid and naik:
        return "Buang atau revisi"
    return "Tinjau ulang sebelum dibuang"


gabung["Keputusan"] = gabung.apply(keputusan, axis=1)
gabung = gabung.merge(kamus_butir, on=["Dimensi", "Butir"])

with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)

try:
    files.download("hasil_uji_validitas_reliabilitas.xlsx")
except NameError:
    print("Tersimpan: hasil_uji_validitas_reliabilitas.xlsx")
gabung[["Dimensi", "Butir", "r hitung", "Corrected r", "Alpha jika dihapus", "Keputusan"]]

Tersimpan: hasil_uji_validitas_reliabilitas.xlsx


,Dimensi,Butir,r hitung,Corrected r,Alpha jika dihapus,Keputusan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.790,0.668,0.744,Pertahankan
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.691,0.585,0.768,Pertahankan
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.627,0.481,0.782,Pertahankan
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.554,0.424,0.791,Pertahankan
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.722,0.495,0.808,Tinjau isi butir
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.800,0.723,0.746,Pertahankan
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.654,0.548,0.775,Pertahankan
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.710,0.605,0.879,Pertahankan
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.780,0.685,0.869,Pertahankan
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.710,0.617,0.877,Pertahankan


## Latihan 1: Butir yang tidak valid

In [9]:
tidak_valid = tabel_validitas[tabel_validitas["Keterangan"] == "Tidak valid"]
print(f"r tabel = {r_tabel:.3f}")
if tidak_valid.empty:
    terendah = tabel_validitas.loc[tabel_validitas["r hitung"].idxmin()]
    print("Tidak ada butir yang tidak valid; semua r hitung > r tabel.")
    print(f"r hitung terendah: {terendah['Dimensi']} {terendah['Butir']} = {terendah['r hitung']}")
else:
    display(tidak_valid)

r tabel = 0.361
Tidak ada butir yang tidak valid; semua r hitung > r tabel.
r hitung terendah: Penggunaan AI pada tugas atau projek kuliah B4 = 0.554


## Latihan 2: Perbandingan kriteria r hitung dan corrected r ≥ 0,30

In [10]:
valid_r = (tabel_validitas["r hitung"] > r_tabel).sum()
valid_rc = (tabel_validitas["Corrected r"] >= 0.30).sum()
print(f"Valid menurut r hitung > {r_tabel:.3f}      : {valid_r} butir")
print(f"Valid menurut corrected r ≥ 0,30        : {valid_rc} butir")

selisih = tabel_validitas.assign(Selisih=(tabel_validitas["r hitung"] - tabel_validitas["Corrected r"]).round(3))
rata_selisih = selisih.groupby("Dimensi", sort=False).agg(jumlah_butir=("Butir", "count"),
                                                         rata_selisih=("Selisih", "mean")).round(3)
rata_selisih

Valid menurut r hitung > 0.361      : 29 butir
Valid menurut corrected r ≥ 0,30        : 29 butir


,jumlah_butir,rata_selisih
Dimensi,,
Penggunaan AI pada tugas atau projek kuliah,7,0.131
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah,7,0.092
Kebiasaan penggunaan teknologi digital,6,0.094
Prokrastinasi akademik,4,0.132
Respon emosional,5,0.069


## Latihan 3: Perbaikan instrumen
Modul meminta membuang satu butir yang **tidak valid**, lalu menjalankan ulang analisis dan membandingkan α dimensi.
Jika tidak ada butir yang tidak valid, tidak ada butir yang dibuang dan seluruh butir dipertahankan.

In [11]:
tidak_valid = tabel_validitas[tabel_validitas["Keterangan"] == "Tidak valid"]
if tidak_valid.empty:
    print("Tidak ada butir yang tidak valid, sehingga tidak ada butir yang dibuang.")
    print("Seluruh", len(tabel_validitas), "butir dipertahankan; α setiap dimensi tetap seperti Langkah 7.")
    display(tabel_reliabilitas)
else:
    buang = tidak_valid.sort_values("r hitung").iloc[0]
    kolom_buang = next(k for k, v in KODE.items() if v == buang["Butir"] and k in dimensi[buang["Dimensi"]])
    dimensi_baru = {k: [c for c in v if c != kolom_buang] for k, v in dimensi.items()}
    rel_sesudah, _ = uji_reliabilitas(num, dimensi_baru)
    print(f"Butir yang dibuang: {buang['Dimensi']} {buang['Butir']}")
    display(tabel_reliabilitas[["Dimensi", "Cronbach's alpha"]].merge(
        rel_sesudah[["Dimensi", "Cronbach's alpha", "Kategori"]], on="Dimensi", suffixes=(" sebelum", " sesudah")))

Tidak ada butir yang tidak valid, sehingga tidak ada butir yang dibuang.
Seluruh 29 butir dipertahankan; α setiap dimensi tetap seperti Langkah 7.


,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.799,Reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.886,Baik
2,Kebiasaan penggunaan teknologi digital,6,0.890,Baik
3,Prokrastinasi akademik,4,0.860,Baik
4,Respon emosional,5,0.928,Sangat baik


## Latihan 4: Pengaruh ukuran sampel (subset acak 20 responden)

In [12]:
num20 = num.sample(20, random_state=1)
val20, r_tabel20 = uji_validitas(num20, dimensi)
rel20, _ = uji_reliabilitas(num20, dimensi)
print(f"n = {len(num)} -> r tabel {r_tabel:.3f} | butir valid {valid_r}")
print(f"n = 20  -> r tabel {r_tabel20:.3f} | butir valid {(val20['Keterangan'] == 'Valid').sum()}")
display(val20[val20["Keterangan"] == "Tidak valid"])
display(tabel_reliabilitas[["Dimensi", "Cronbach's alpha"]].merge(
    rel20[["Dimensi", "Cronbach's alpha", "Kategori"]], on="Dimensi", suffixes=(f" n={len(num)}", " n=20")))

n = 30 -> r tabel 0.361 | butir valid 29
n = 20  -> r tabel 0.444 | butir valid 28


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.421,0.256,0.0647,Tidak valid


,Dimensi,Cronbach's alpha n=30,Cronbach's alpha n=20,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,0.799,0.713,Reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,0.886,0.925,Sangat baik
2,Kebiasaan penggunaan teknologi digital,0.890,0.917,Sangat baik
3,Prokrastinasi akademik,0.860,0.801,Baik
4,Respon emosional,0.928,0.941,Sangat baik
